# Customer Churn Prediction & Retention Modeling

This notebook uses IBM's public **Telco Customer Churn** sample dataset.

**Business question:** Which customers are most likely to churn, and which customer/service characteristics are most associated with that risk?

The goal is to combine predictive modeling with a retention-prioritization use case rather than treating classification accuracy as the only objective.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.inspection import permutation_importance

df = pd.read_csv("../data/Telco-Customer-Churn.csv")
df.head()


## 1. Data Cleaning


In [ ]:
df.info()

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["ChurnFlag"] = df["Churn"].map({"Yes": 1, "No": 0})

print("Rows:", len(df))
print("Missing TotalCharges:", df["TotalCharges"].isna().sum())
print("Churn rate:", df["ChurnFlag"].mean())


## 2. Exploratory Churn Patterns


In [ ]:
(
    df.groupby("Contract")["ChurnFlag"]
      .mean()
      .sort_values(ascending=False)
      .plot(kind="bar", title="Churn Rate by Contract Type")
)
plt.ylabel("Churn Rate")
plt.tight_layout()
plt.show()


In [ ]:
(
    df.groupby("InternetService")["ChurnFlag"]
      .mean()
      .sort_values(ascending=False)
      .plot(kind="bar", title="Churn Rate by Internet Service")
)
plt.ylabel("Churn Rate")
plt.tight_layout()
plt.show()


In [ ]:
(
    df.groupby("PaymentMethod")["ChurnFlag"]
      .mean()
      .sort_values(ascending=False)
      .plot(kind="bar", title="Churn Rate by Payment Method")
)
plt.ylabel("Churn Rate")
plt.tight_layout()
plt.show()


## 3. Prepare the Modeling Pipeline


In [ ]:
X = df.drop(columns=["customerID", "Churn", "ChurnFlag"])
y = df["ChurnFlag"]

numeric = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
categorical = [c for c in X.columns if c not in numeric]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)


## 4. Compare Models with Cross-Validation


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=350, max_depth=12, min_samples_leaf=4,
        class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []
trained = {}

for name, estimator in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])

    cv_auc = cross_val_score(
        pipe, X_train, y_train,
        scoring="roc_auc", cv=cv, n_jobs=-1
    )

    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba),
        "CV ROC-AUC": cv_auc.mean()
    })

    trained[name] = pipe

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False)
results_df


## 5. Best Model Evaluation


In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = trained[best_name]
print("Best model:", best_name)

ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test)
plt.title(f"Confusion Matrix — {best_name}")
plt.show()

RocCurveDisplay.from_estimator(best_model, X_test, y_test)
plt.title(f"ROC Curve — {best_name}")
plt.show()


## 6. Model-Agnostic Feature Importance


In [ ]:
perm = permutation_importance(
    best_model, X_test, y_test,
    scoring="roc_auc",
    n_repeats=8,
    random_state=42,
    n_jobs=-1
)

importance = (
    pd.DataFrame({
        "Feature": X_test.columns,
        "Importance": perm.importances_mean
    })
    .sort_values("Importance", ascending=False)
)

importance.head(10)


## 7. Business Interpretation

Rather than using the model as an automatic decision-maker, the prediction can support **retention prioritization**.

A retention team could use churn probabilities to:
- identify customers who may need proactive outreach;
- investigate common service or billing friction;
- test contract or service interventions;
- compare whether targeted retention programs actually reduce churn.

In a real production system, model monitoring, privacy review, fairness analysis, and controlled experimentation would be required before operational use.
